# PlatoSim3 paper: PSF

This notebook is used to generate the input plots of the PSF models shown in the PlatoSim3 paper; Jannsen et al. (2023).

In [1]:
# Alow changes to the PlatoSim code outside this notebook
%load_ext autoreload
%autoreload 2

# Configure figure in notebook
%matplotlib notebook

In [2]:
import os
import h5py
import scipy
import wotan
import numpy as np
import pandas as pd
from tqdm import tqdm

from matplotlib import pyplot as plt
import matplotlib.colors  as colors
import matplotlib.patches as patches

from astropy.io import fits
from astropy import units as u
from astropy.coordinates import SkyCoord

from scipy.signal import periodogram
from scipy.ndimage import median_filter
from scipy.interpolate import interp2d

import transitleastsquares as tls

# PlatoSim
import platosim.referenceFrames as rf
import platosim.plot            as pt
import platosim.utilities       as ut
from platosim.simulation   import Simulation
from platosim.simfile      import SimFile
from platosim.lightcurve   import LightCurve
from platosim.validation import switchOffAllEffects
from platosim.matplotlibrc import setup_paper
setup_paper()

# Constants
day = 86400

---
## Synthetic PSF: Zemax and Analytic
---

In [ ]:
# Initialise PlatoSim
outputFileName = "output_PSF_test"
sim = Simulation(outputFileName, outputDir=os.getcwd())
pixelSize   = sim["CCD/PixelSize"]                            # [um]
focalLength = sim["Camera/FocalLength/ConstantValue"] * 1000  # [mm]
ccdAngles   = sim["CCDPositions/Orientation"]

# Set global parameters for this section
fs = 25
cm = 'gist_stern'
res_z = 64   # Zemax    PSF resolution [subpixel/pixel]
res_a = 128  # Analytic PSF resolution [subpixel/pixel]
sub2pix_z = [0, res_z, int(res_z*2), int(res_z*3), int(res_z*4)]
sub2pix_a = [0, res_a, int(res_a*2), int(res_a*3), int(res_a*4)]
ticks =["0", "1", "2", "3", "4"]

# Constants
deg2mm = rf.focalPlaneCoordinatesFromGnomonicRadialDistance(np.deg2rad(1), focalLength)[0]
deg2pix = 240

xFP, yFP   = rf.focalPlaneCoordinatesFromGnomonicRadialDistance(np.deg2rad(16), focalLength, inPlaneRotation=np.deg2rad(30))
xCCD, yCCD = rf.focalPlaneToPixelCoordinates(xFP, yFP, pixelSize, ccdZeroPointX=0, ccdZeroPointY=0, CCDangle=0)
print(xFP, yFP)
print(xCCD, yCCD)

x1 = 2.13
x2 = 13.165
alpha = rf.gnomonicRadialDistanceFromOpticalAxis(x1*deg2mm, x1*deg2mm, focalLength)
print(xCCD, yCCD)
print(np.rad2deg(alpha))

In [ ]:
# Define small function to generate a PSF
def runsim(sim, ccdCode, xCCD, yCCD, N=8):
    
    # Observation
    sim["ObservingParameters/NumExposures"] = 1

    # Camera
    sim["Camera/IncludePointLikeGhosts"] = False 

    # PSF
    sim["PSF/MappedFromFile/IncludeChargeDiffusion"] = True

    # Control HDF5
    sim["ControlHDF5Content/WriteStarPositions"]     = True
    sim["ControlHDF5Content/WriteDiffusedPSF"]       = True
    sim["ControlHDF5Content/WriteHighResolutionPSF"] = True

    # Set subfiled
    # Note setting the coordinates only will only allow to 
    setSubfield = sim.setSubfieldAroundPixelCoordinates(ccdCode=ccdCode, 
                                                        xCCDpixel=xCCD, yCCDpixel=yCCD, 
                                                        subfieldSizeX=8, subfieldSizeY=8)
    # Run simulation
    sim.run(removeOutputFile=True)

### Test case: Analytic

In [ ]:
# Check coordinate from Carsten's plot
ccdCode = "1"
xCCDdeg = x1 + 5 * 2
yCCDdeg = x1 + 5 * 1
xCCD = xCCDdeg * deg2pix
yCCD = yCCDdeg * deg2pix 

# Initialise PlatoSim
outputFileName = "output_AnalyticPSF_test"
sim = Simulation(outputFileName, outputDir=os.getcwd())
sim["PSF/Model"] = "AnalyticNonGaussian"
runsim(sim, ccdCode, xCCD, yCCD)

In [ ]:
# Plot Zemax high res PSFs
fig, ax = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(6,6))
fig.subplots_adjust(hspace=0.02, wspace=0)
# Load correct PSF
string = "output_AnalyticPSF_test"
f   = h5py.File(f'{os.getcwd()}/{string}.hdf5', 'r')
psf = np.array(f["PSF/highResPSF"])[2*res_a:6*res_a, 2*res_a:6*res_a] 
# Rotate psf
angles = [180, 90, 0, 270]
# angles = [270, 180, 270, 0]
rotate = angles[int(ccdCode)-1]
psf_rot = scipy.ndimage.rotate(psf, rotate)
# Plot PSF
im = ax.imshow(psf_rot, cmap=cm, interpolation='nearest', origin='lower')
print(rotate)

### Test case: Zemax

In [ ]:
# Coordinate corresponding to (-2.5, -2.5) in Carstens plot
# Warning! Zemax is alway using a CCD number lower than the Analytic!
ccdCode = "2"
xFPdeg = 2.5 + 5 * 3
yFPdeg = 2.5 + 5 * 1
xFPmm = xFPdeg * deg2mm
yFPmm = yFPdeg * deg2mm
xCCD = xFPdeg * deg2pix
yCCD = yFPdeg * deg2pix 
print(xCCD, yCCD)

# Initialise PlatoSim
outputFileName = "output_ZemaxPSF_test"
sim = Simulation(outputFileName, outputDir=os.getcwd())
sim["PSF/Model"] = "MappedFromFile"
runsim(sim, ccdCode, xCCD, yCCD)

In [ ]:
# Plot Zemax high res PSFs
fig, ax = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(6,6))
fig.subplots_adjust(hspace=0.02, wspace=0)
# Load correct PSF
string = "output_ZemaxPSF_test"
f   = h5py.File(f'{os.getcwd()}/{string}.hdf5', 'r')
psf = np.array(f["PSF/rotatedPSF"])[2*res_z:6*res_z, 2*res_z:6*res_z] 
# Rotate psf
angles = [180, 90, 0, 270]
rotate = angles[int(ccdCode)-1]
psf_rot = scipy.ndimage.rotate(psf, rotate)
# Plot PSF
im = ax.imshow(psf_rot, cmap=cm, interpolation='nearest', origin='lower')

### Analytic PSF

In [ ]:
# Coordinate corresponding to (-2.5, -2.5) in Carstens plot
ccdCode = "1"
xFPdeg = x1 + 5 * 0
yFPdeg = x1 + 5 * 3
xFPmm = xFPdeg * deg2mm
yFPmm = yFPdeg * deg2mm
xCCD = xFPdeg * deg2pix
yCCD = yFPdeg * deg2pix 

# Initialise PlatoSim
outputFileName = "output_AnalyticPSF_lowAlpha"
sim = Simulation(outputFileName, outputDir=os.getcwd())

# Run simulation
sim["PSF/Model"] = "AnalyticNonGaussian"
runsim(sim, ccdCode, xCCD, yCCD)

In [ ]:
# Coordinate corresponding to (-2.5, -2.5) in Carstens plot
ccdCode = "1"
xFPdeg = x1 + 5 * 2
yFPdeg = x1 + 5 * 1
xFPmm = xFPdeg * deg2mm
yFPmm = yFPdeg * deg2mm
xCCD = xFPdeg * deg2pix
yCCD = yFPdeg * deg2pix 

# Initialise PlatoSim
outputFileName = "output_AnalyticPSF_highAlpha"
sim = Simulation(outputFileName, outputDir=os.getcwd())

# Run simulation
sim["PSF/Model"] = "AnalyticNonGaussian"
runsim(sim, ccdCode, xCCD, yCCD)

In [ ]:
# Load high resolution and diffused PSFs
f0 = h5py.File(f'{os.getcwd()}/output_AnalyticPSF_lowAlpha.hdf5', 'r')
psf0a = np.array(f0["PSF/highResPSF"] )[2*res_a:6*res_a, 2*res_a:6*res_a]  # We only want the inner 4 x 4 pixels
psf0d = np.array(f0["PSF/diffusedPSF"])[2*res_a:6*res_a, 2*res_a:6*res_a]

f1 = h5py.File(f'{os.getcwd()}/output_AnalyticPSF_highAlpha.hdf5', 'r')
psf1a = np.array(f1["PSF/highResPSF"] )[2*res_a:6*res_a, 2*res_a:6*res_a]  
psf1d = np.array(f1["PSF/diffusedPSF"])[2*res_a:6*res_a, 2*res_a:6*res_a]

# Combine for plot
psfAnalytic = [psf0a, psf1a, psf0d, psf1d]

# Normalize high-res and diffused PSFs w.r.t. each other, respectuvely
vmax0 = np.max(psf0a)
vmax1 = np.max(psf0d)

In [ ]:
fig, ax = plt.subplots(2, 2, sharex=True, sharey=True, figsize=(9,9))
fig.subplots_adjust(hspace=0.07, wspace=0)

# Create plots
for i, psf, axes in zip(range(4), psfAnalytic, ax.flat):
    # Rotate psf
    angles = [180, 90, 0, 270]
    rotate = angles[int(ccdCode)-1]
    psf_rot = scipy.ndimage.rotate(psf, rotate)
    
    if i in (0, 1):
        im0 = axes.imshow(psf_rot, cmap=cm, vmin=0, vmax=vmax0, interpolation='nearest', origin='lower')
    elif i in (2, 3):
        im1 = axes.imshow(psf_rot, cmap=cm, vmin=0, vmax=vmax1, interpolation='nearest', origin='lower')

    # Settings
    axes.set_xticks(sub2pix_a)
    axes.set_yticks(sub2pix_a)
    axes.set_xticklabels(ticks)
    axes.set_yticklabels(ticks)

# Upper Colorbar
cbarax = fig.add_axes([0.91, 0.51, 0.03, 0.34])
cbar0 = plt.colorbar(im0, orientation='vertical', cax=cbarax, extend='max', cmap=cm)                                                                                                                          
cbar0.ax.tick_params()
# Colorbar scientific notation
cbar0.formatter.set_powerlimits((0, 0))
locs = pt.moveColorbarExponent(x_offs=0.0, y_offs=1.15, dig=1, side='left')

# Lower Colorbar
cbarax = fig.add_axes([0.91, 0.11, 0.03, 0.34])
cbar1 = plt.colorbar(im1, orientation='vertical', cax=cbarax, extend='max', cmap=cm)                                                                                                                          
cbar1.ax.tick_params()
# Colorbar scientific notation
cbar1.formatter.set_powerlimits((0, 0))
locs = pt.moveColorbarExponent(x_offs=0.0, y_offs=1.15, dig=1, side='left')

# Set all text
fig.suptitle("b) Analytic model", y=0.93, fontsize=22)
fig.text(0.5, 0.05,  r'Pixel column, $i$', ha='center')
fig.text(0.05, 0.5,  r'Pixel row, $j$', va='center', rotation='vertical')
fig.text(0.32, 0.83, r'High resolution: $\alpha=3^{\circ}$',     ha='center', color='white')
fig.text(0.70, 0.83, r'High resolution: $\alpha=18^{\circ}$',    ha='center', color='white')
fig.text(0.32, 0.44, r'Diffused: $\alpha=3^{\circ}$',  ha='center', color='white')
fig.text(0.70, 0.44, r'Diffused: $\alpha=18^{\circ}$', ha='center', color='white');

# Save figure
fig.savefig('psfAnalytic.png', bbox_inches='tight', dpi=200)

### Zemax PSF

In [ ]:
# Coordinate corresponding to (-2.5, -2.5) in Carstens plot
ccdCode = "4"
xFPdeg = x1 + 5 * 0
yFPdeg = x1 + 5 * 3
xFPmm = xFPdeg * deg2mm
yFPmm = yFPdeg * deg2mm
xCCD = xFPdeg * deg2pix
yCCD = yFPdeg * deg2pix 
alpha = rf.gnomonicRadialDistanceFromOpticalAxis(12.5*deg2mm, 7.5*deg2mm, focalLength)
print(xCCD, yCCD)
print(np.rad2deg(alpha))

# Initialise PlatoSim
outputFileName = "output_ZemaxPSF_lowAlpha"
sim = Simulation(outputFileName, outputDir=os.getcwd())

# Run simulation
sim["PSF/Model"] = "MappedFromFile"
runsim(sim, ccdCode, xCCD, yCCD)

In [ ]:
# Coordinate corresponding to (-12.5, -7.5) in Carstens plot
ccdCode = "4"
xFPdeg = x1 + 5 * 2
yFPdeg = x1 + 5 * 1
xFPmm = xFPdeg * deg2mm
yFPmm = yFPdeg * deg2mm
xCCD = xFPdeg * deg2pix
yCCD = yFPdeg * deg2pix 

# Initialise PlatoSim
outputFileName = "output_ZemaxPSF_highAlpha"
sim = Simulation(outputFileName, outputDir=os.getcwd())

# Run simulation
sim["PSF/Model"] = "MappedFromFile"
runsim(sim, ccdCode, xCCD, yCCD)

In [ ]:
# Load high resolution and diffused PSFs
f0 = h5py.File(f'{os.getcwd()}/output_ZemaxPSF_lowAlpha.hdf5', 'r')
psf0z = np.array(f0["PSF/rotatedPSF"] )[2*res_z:6*res_z, 2*res_z:6*res_z]  # We only want the inner 4 x 4 pixels
psf0d = np.array(f0["PSF/diffusedPSF"])[2*res_z:6*res_z, 2*res_z:6*res_z]

f1 = h5py.File(f'{os.getcwd()}/output_ZemaxPSF_highAlpha.hdf5', 'r')
psf1z = np.array(f1["PSF/rotatedPSF"] )[2*res_z:6*res_z, 2*res_z:6*res_z]  
psf1d = np.array(f1["PSF/diffusedPSF"])[2*res_z:6*res_z, 2*res_z:6*res_z]

# Combine for plot
psfZemax = [psf0z, psf1z, psf0d, psf1d] 

# Normalize high-res and diffused PSFs w.r.t. each other, respectuvely
vmax0 = np.max(psf0z)
vmax1 = np.max(psf0d)

In [ ]:
# Plot figure
fig, ax = plt.subplots(2, 2, sharex=True, sharey=True, figsize=(9,9))
fig.subplots_adjust(hspace=0.07, wspace=0)

# Create plots
for i, psf, axes in zip(range(4), psfZemax, ax.flat):
    # Rotate psf
    angles = [180, 90, 0, 270]
    rotate = angles[int(ccdCode)-1]
    psf_rot = scipy.ndimage.rotate(psf, rotate)
    if i in (0, 1):
        im0 = axes.imshow(psf_rot, cmap=cm, vmin=0, vmax=vmax0, interpolation='nearest', origin='lower')
    elif i in (2, 3):
        im1 = axes.imshow(psf_rot, cmap=cm, vmin=0, vmax=vmax1, interpolation='nearest', origin='lower')

    # Settings
    axes.set_xticks(sub2pix_z)
    axes.set_yticks(sub2pix_z)
    axes.set_xticklabels(ticks)
    axes.set_yticklabels(ticks)

# Upper Colorbar
cbarax = fig.add_axes([0.91, 0.51, 0.03, 0.34])
cbar0 = plt.colorbar(im0, orientation='vertical', cax=cbarax, extend='max', cmap=cm)                                                                                                                          
cbar0.ax.tick_params()
# Colorbar scientific notation
cbar0.formatter.set_powerlimits((0, 0))
locs = pt.moveColorbarExponent(x_offs=0.0, y_offs=1.15, dig=1, side='left')

# Lower Colorbar
cbarax = fig.add_axes([0.91, 0.11, 0.03, 0.34])
cbar1 = plt.colorbar(im1, orientation='vertical', cax=cbarax, extend='max', cmap=cm)                                                                                                                          
cbar1.ax.tick_params()
# Colorbar scientific notation
cbar1.formatter.set_powerlimits((0, 0))
locs = pt.moveColorbarExponent(x_offs=0.0, y_offs=1.15, dig=1, side='left')

# Set all text
fig.suptitle("a) Zemax model", y=0.93, fontsize=22)
fig.text(0.5, 0.05, r'Pixel column, $i$', ha='center')
fig.text(0.05, 0.5, r'Pixel row, $j$', va='center', rotation='vertical')
fig.text(0.32, 0.83, r'High resolution: $\alpha=3^{\circ}$',     ha='center', color='white')
fig.text(0.70, 0.83, r'High resolution: $\alpha=18^{\circ}$',    ha='center', color='white')
fig.text(0.32, 0.44, r'Diffused: $\alpha=3^{\circ}$',  ha='center', color='white')
fig.text(0.70, 0.44, r'Diffused: $\alpha=18^{\circ}$', ha='center', color='white');

# Save figure
fig.savefig('psfZemax.png', bbox_inches='tight', dpi=200)

---
## PSF over full FP
---

In [ ]:
# Matrix to calculate the PSFs
xFP = np.array([  0.0,   0.0, -7.5, -2.5, 2.5, 7.5,  0.0,  0.0,
                  0.0, -12.5, -7.5, -2.5, 2.5, 7.5, 12.5,  0.0,
                -17.5, -12.5, -7.5, -2.5, 2.5, 7.5, 12.5, 17.5,
                -17.5, -12.5, -7.5, -2.5, 2.5, 7.5, 12.5, 17.5,
                -17.5, -12.5, -7.5, -2.5, 2.5, 7.5, 12.5, 17.5,
                -17.5, -12.5, -7.5, -2.5, 2.5, 7.5, 12.5, 17.5,
                  0.0, -12.5, -7.5, -2.5, 2.5, 7.5, 12.5,  0.0,
                  0.0,   0.0, -7.5, -2.5, 2.5, 7.5,  0.0,  0.0])
yFP = np.array([  0.0,   0.0,  17.5,  17.5,  17.5,  17.5,   0.0,   0.0,
                  0.0,  12.5,  12.5,  12.5,  12.5,  12.5,  12.5,   0.0,
                  7.5,   7.5,   7.5,   7.5,   7.5,   7.5,   7.5,   7.5,
                  2.5,   2.5,   2.5,   2.5,   2.5,   2.5,   2.5,   2.5,
                 -2.5,  -2.5,  -2.5,  -2.5,  -2.5,  -2.5,  -2.5,  -2.5,
                 -7.5,  -7.5,  -7.5,  -7.5,  -7.5,  -7.5,  -7.5,  -7.5,
                  0.0, -12.5, -12.5, -12.5, -12.5, -12.5, -12.5,   0.0,
                  0.0,   0.0, -17.5, -17.5, -17.5, -17.5,   0.0,   0.0])
alpha = np.sqrt(xFP**2 + yFP**2)
inrot = np.rad2deg(np.arctan(yFP/xFP))
n = len(alpha)

In [ ]:
i = 10
ccdCode = "1"
deg2mm = rf.focalPlaneCoordinatesFromGnomonicRadialDistance(np.deg2rad(1), focalLength)[0]
xFP_i = xFP[i] * deg2mm
yFP_i = yFP[i] * deg2mm

In [ ]:
# Initialise PlatoSim
outputFileName = "output_AnalyticPSFfullFP_test"
sim = Simulation(outputFileName, outputDir=os.getcwd())
pixelSize     = sim["CCD/PixelSize"]
ccdZeroPointX = sim['CCDPositions/OriginOffsetX'][int(ccdCode)]
ccdZeroPointY = sim['CCDPositions/OriginOffsetY'][int(ccdCode)]
CCDangle      = np.deg2rad(sim['CCDPositions/Orientation'][int(ccdCode)])

In [ ]:
# From focal plane coordinates to CCD coordinates
xCCD, yCCD = rf.focalPlaneToPixelCoordinates(xFP_i, yFP_i, pixelSize, ccdZeroPointX, ccdZeroPointY, CCDangle)
xCCD, yCCD

In [ ]:
# Run simulation
sim["PSF/Model"] = "AnalyticNonGaussian"
runsim(sim, ccdCode, xCCD, yCCD)

In [ ]:
# Plot Zemax high res PSFs
fig, ax = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(6,6))
fig.subplots_adjust(hspace=0.02, wspace=0)
# Load correct PSF
f   = h5py.File(f'{os.getcwd()}/{outputFileName}.hdf5', 'r')
psf = np.array(f["PSF/highResPSF"])[2*res_a:6*res_a, 2*res_a:6*res_a] 
# Rotate psf
angles = [180, 90, 0, 270]
# angles = [270, 180, 270, 0]
rotate = angles[int(ccdCode)-1]
psf_rot = scipy.ndimage.rotate(psf, rotate)
# Plot PSF
im = ax.imshow(psf, cmap=cm, interpolation='nearest', origin='lower')
print(rotate)

In [ ]:
# Run simulation
sim.run(removeOutputFile=True)

# Plot Zemax high res PSFs
fig, ax = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(6,6))
fig.subplots_adjust(hspace=0.02, wspace=0)
# Load correct PSF
string = "output_AnalyticPSFfullFP_test"
f   = h5py.File(f'{os.getcwd()}/{string}.hdf5', 'r')
psf = np.array(f["PSF/highResPSF"])[2*res_a:6*res_a, 2*res_a:6*res_a] 
# Plot PSF
im = ax.imshow(psf, cmap="gnuplot2", interpolation='nearest', origin='lower')

In [ ]:
# From focal plane coordinates to CCD coordinates
# xFP_rad = np.deg2rad(-12.5)
# yFP_rad = np.deg2rad(-12.5)
# xFP = focalLength * np.tan(xFP_rad)
# yFP = focalLength * np.tan(yFP_rad)
# xCCD, yCCD = rf.focalPlaneToPixelCoordinates(xFP, yFP, 
#                                              pixelSize=sim["CCD/PixelSize"], 
#                                              ccdZeroPointX=0, ccdZeroPointY=0, 
#                                              CCDangle=ang)

In [ ]:
# Loop over each simulation
for i in tqdm(range(n), bar_format=ut.tqdm_bar_format()):

    # Select values
    alpha_i = alpha[i]
    inrot_i = inrot[i]
    xFP_i   = xFP[i]
    yFP_i   = yFP[i]
    
    # Don't simulate empty spots
    if alpha_i == 0:
        pass
    else:
        
        # From alpha to focal plane coordinates
        if i in (4,5,6,7,12,13,14,15,20,21,22,23,28,29,30,31):
            ccdAngle = 0
        elif i in (0,1,2,3,8,9,10,11,16,17,18,19,24,25,26,27):
            ccdAngle = 90
            inrot_i  = 90 + inrot_i
        elif i in (32,33,34,35,40,41,42,43,48,49,50,51,56,57,58,59):
            ccdAngle = 180
            inrot_i  = 180 + inrot_i
        elif i in (36,37,38,39,44,45,46,47,52,53,54,55,60,61,62,63):
            ccdAngle = 270
            inrot_i  = 270 + inrot_i
            
        # Initialise PlatoSim
        outputFileName = "output_AnalyticPSFfullFP_num" + f"{i}".zfill(2)
        sim = Simulation(outputFileName, outputDir=os.getcwd())

        focalLength = sim["Camera/FocalLength/ConstantValue"] * 1000
        xFP_pos, yFP_pos = rf.focalPlaneCoordinatesFromGnomonicRadialDistance(np.deg2rad(alpha_i), focalLength, 
                                                                              inPlaneRotation=np.deg2rad(inrot_i))

        # From focal plane coordinates to CCD coordinates
        pixelSize  = sim["CCD/PixelSize"]
        xCCD, yCCD = rf.focalPlaneToPixelCoordinates(xFP_pos, yFP_pos, pixelSize, 
                                                     ccdZeroPointX=0, ccdZeroPointY=0, 
                                                     CCDangle=ccdAngle)

        # Observation
        sim["ObservingParameters/NumExposures"] = 1

        # Subfield (note y here is row and x is column)
        xzero = sim["SubField/ZeroPointRow"]    = int(yCCD)
        yzero = sim["SubField/ZeroPointColumn"] = int(xCCD)
        xsub  = sim["SubField/NumColumns"]      = 8
        ysub  = sim["SubField/NumRows"]         = 8

        # Camera
        sim["Camera/IncludePointLikeGhosts"] = False
    
        # PSF
        sim["PSF/Model"] = "AnalyticNonGaussian"
        sim["PSF/MappedFromFile/IncludeChargeDiffusion"] = True

        # Control HDF5
        sim["ControlHDF5Content/WriteStarPositions"]     = True
        sim["ControlHDF5Content/WriteDiffusedPSF"]       = True
        sim["ControlHDF5Content/WriteHighResolutionPSF"] = True

        # Create star catalogue (path to starcat is automatically by function)
        xpos = np.array([int(xsub/2.) + xzero])
        ypos = np.array([int(ysub/2.) + yzero])
        mag  = np.array([10.0])
        ID   = np.array([1])
        starcat = f"{os.getcwd()}/starcat.txt"
        sim.createStarCatalogFileFromPixelCoordinates(xpos, ypos, mag, ID, starcat)

        # Run simulation
        sim.run(removeOutputFile=True)

In [ ]:
# Plot Zemax high res PSFs
fig, ax = plt.subplots(8, 8, sharex=True, sharey=True, figsize=(14,14))
fig.subplots_adjust(hspace=0.02, wspace=0)
fig.suptitle("Analytic High Resolution PSFs", y=0.9, fontsize=20)

# Create plots
for i, axes in zip(range(n), ax.flat):

    # Load high resolution and diffused PSFs
    xFP_i = xFP[i]
    yFP_i = yFP[i]
    
    # Don't simulate empty spots
    if xFP_i == 0:
        axes.axis('off')
    else:
        
        # Load correct PSF
        string = "output_AnalyticPSFfullFP_num" + f"{i}".zfill(2)
        f   = h5py.File(f'{os.getcwd()}/{string}.hdf5', 'r')
        psf = np.array(f["PSF/highResPSF"])[2*res_a:6*res_a, 2*res_a:6*res_a]
    
        # Plot PSF
        im = axes.imshow(psf, cmap=cm, interpolation='nearest', origin='lower')
    
        # Setting
        text = f"{xFP_i:.1f}" + r"$^{\circ}$, " + f"{yFP_i:.1f}" + r"$^{\circ}$"
        axes.text(0.5, 0.9, text, color='white', fontsize=13, 
                  horizontalalignment='center', verticalalignment='center', transform=axes.transAxes)
    axes.set_xticks([])
    axes.set_yticks([])
    axes.set_xticklabels([])
    axes.set_yticklabels([])

# Save figure
fig.savefig('psfAnalyticHigResFullFP.pdf', bbox_inches='tight', dpi=200)

In [ ]:
# Loop over each simulation
for i in tqdm(range(n), bar_format=ut.tqdm_bar_format()):

    # Select values
    alpha_i = alpha[i]
    inrot_i = inrot[i]
    
    # Don't simulate empty spots
    if alpha_i == 0:
        pass
    else: 
        # Initialise PlatoSim
        outputFileName = ("output_ZemaxPSFfullFP_num" + f"{i}".zfill(2) + 
                          "_alpha{:.1f}_inrot{:.1f}".format(alpha_i, inrot_i))
        sim = Simulation(outputFileName, outputDir=os.getcwd())

        # From alpha to focal plane coordinates
        if i in (0,1,2,3,8,9,10,11,16,17,18,19):
            inrot_i = 180 - inrot_i
        focalLength = sim["Camera/FocalLength/ConstantValue"] * 1000
        xFP, yFP = rf.focalPlaneCoordinatesFromGnomonicRadialDistance(np.deg2rad(alpha_i), focalLength, 
                                                                      inPlaneRotation=np.deg2rad(inrot_i))

        # From focal plane coordinates to CCD coordinates
        pixelSize  = sim["CCD/PixelSize"]
        xCCD, yCCD = rf.focalPlaneToPixelCoordinates(xFP, yFP, pixelSize, 
                                                 ccdZeroPointX=0, ccdZeroPointY=0, CCDangle=0)

        # Observation
        sim["ObservingParameters/NumExposures"] = 1

        # Subfield (note y here is row and x is column)
        xzero = sim["SubField/ZeroPointRow"]    = int(yCCD)
        yzero = sim["SubField/ZeroPointColumn"] = int(xCCD)
        xsub  = sim["SubField/NumColumns"]      = 8
        ysub  = sim["SubField/NumRows"]         = 8

        # Camera
        sim["Camera/IncludePointLikeGhosts"] = False
    
        # PSF
        sim["PSF/Model"] = "MappedFromFile"
        sim["PSF/MappedFromFile/IncludeChargeDiffusion"] = True

        # Control HDF5
        sim["ControlHDF5Content/WriteStarPositions"]     = True
        sim["ControlHDF5Content/WriteDiffusedPSF"]       = True
        sim["ControlHDF5Content/WriteHighResolutionPSF"] = True

        # Create star catalogue (path to starcat is automatically by function)
        xpos = np.array([int(xsub/2.) + xzero])
        ypos = np.array([int(ysub/2.) + yzero])
        mag  = np.array([10.0])
        ID   = np.array([1])
        starcat = f"{os.getcwd()}/starcat.txt"
        sim.createStarCatalogFileFromPixelCoordinates(xpos, ypos, mag, ID, starcat)

        # Run simulation
        sim.run(removeOutputFile=True)

In [ ]:
# Plot Zemax high res PSFs
fig, ax = plt.subplots(8, 8, sharex=True, sharey=True, figsize=(14,14))
fig.subplots_adjust(hspace=0.02, wspace=0)
fig.suptitle("Zemax High Resolution PSFs", y=0.9, fontsize=20)

# Create plots
for i, axes in zip(range(n), ax.flat):

    # Load high resolution and diffused PSFs
    alpha_i = alpha[i]
    inrot_i = inrot[i]
    
    # Don't simulate empty spots
    if alpha_i == 0:
        pass
    else: 
        string = "output_ZemaxPSFfullFP_num" + f"{i}".zfill(2) + "_alpha{:.1f}_inrot{:.1f}".format(alpha_i, inrot_i)
        f   = h5py.File(f'{os.getcwd()}/{string}.hdf5', 'r')
        psf = np.array(f["PSF/rotatedPSF"])[2*res_z:6*res_z, 2*res_z:6*res_z]  # We only want the inner 4 x 4 pixels
    
        # Plot PSF
        im = axes.imshow(psf, cmap=cm, interpolation='nearest', origin='lower')
    
        # Setting
        text = f"{alpha_i:.1f}" + r"$^{\circ}$, " + f"{inrot_i:.1f}" + r"$^{\circ}$"
        axes.text(0.5, 0.9, text, color='white', fontsize=13, 
                  horizontalalignment='center', verticalalignment='center', transform=axes.transAxes)
    axes.set_xticks([])
    axes.set_yticks([])
    axes.set_xticklabels([])
    axes.set_yticklabels([])

# Save figure
fig.savefig('psfZemaxHigResFullFP.pdf', bbox_inches='tight', dpi=200)

---
## Qudrant of FP
---

In [ ]:
# Matrix to calculate the PSFs
alpha = [14.6, 14.7, 15.2, 15.9, 16.8, 17.9, 19.2, 20.6,
         12.5, 12.7, 13.2, 14.0, 15.0, 16.3, 17.7, 19.2,
         10.4, 10.6, 11.2, 12.1, 13.3, 14.7, 16.3, 17.9,
          8.3,  8.6,  9.3, 10.4, 11.8, 13.3, 15.0, 16.8,
          6.2,  6.6,  7.5,  8.8, 10.4, 12.1, 14.0, 15.9,
          4.2,  4.7,  5.9,  7.5,  9.3, 11.2, 13.2, 15.2, 
          2.1,  2.9,  4.7,  6.6,  8.6, 10.6, 12.7, 14.7,
          0.0,  2.1,  4.2,  6.2,  8.3, 10.4, 12.5, 14.6]
inrot = [90.0, 81.9, 74.1, 66.8, 60.3, 54.5, 49.4, 45.0,
         90.0, 80.5, 71.6, 63.4, 56.3, 50.2, 45.0, 40.6,
         90.0, 78.7, 68.2, 59.0, 51.3, 45.0, 39.8, 35.5,
         90.0, 76.0, 63.4, 53.1, 45.0, 38.7, 33.7, 29.7,
         90.0, 71.6, 56.3, 45.0, 36.9, 31.0, 26.6, 23.2,
         90.0, 63.4, 45.0, 33.7, 26.6, 21.8, 18.4, 15.9,
         90.0, 45.0, 26.0, 18.4, 14.0, 11.3,  9.5,  8.1,
          0.0,  0.0,  0.0,  0.0,  0.0,  0.0,  0.0, 0.0]
n = len(alpha)

In [ ]:
# Loop over each simulation
for i in tqdm(range(n), bar_format=ut.tqdm_bar_format()):

    # Select values
    alpha_i = alpha[i]
    inrot_i = inrot[i]
    
    # Initialise PlatoSim
    outputFileName = ("output_ZemaxPSF_num" + f"{i}".zfill(2) + "_alpha{:.1f}_inrot{:.1f}".format(alpha_i, inrot_i))
    sim = Simulation(outputFileName, outputDir=os.getcwd())

    # From alpha to focal plane coordinates
    focalLength = sim["Camera/FocalLength/ConstantValue"] * 1000
    xFP, yFP = rf.focalPlaneCoordinatesFromGnomonicRadialDistance(np.deg2rad(alpha_i), focalLength, 
                                                                  inPlaneRotation=np.deg2rad(inrot_i))

    # From focal plane coordinates to CCD coordinates
    pixelSize  = sim["CCD/PixelSize"]
    xCCD, yCCD = rf.focalPlaneToPixelCoordinates(xFP, yFP, pixelSize, 
                                                 ccdZeroPointX=0, ccdZeroPointY=0, CCDangle=0)

    # Observation
    sim["ObservingParameters/NumExposures"] = 1

    # Subfield (note y here is row and x is column)
    xzero = sim["SubField/ZeroPointRow"]    = int(yCCD)
    yzero = sim["SubField/ZeroPointColumn"] = int(xCCD)
    xsub  = sim["SubField/NumColumns"]      = 8
    ysub  = sim["SubField/NumRows"]         = 8

    # Camera
    sim["Camera/IncludePointLikeGhosts"] = False
    
    # PSF
    sim["PSF/Model"] = "MappedFromFile"
    sim["PSF/MappedFromFile/IncludeChargeDiffusion"] = True

    # Control HDF5
    sim["ControlHDF5Content/WriteStarPositions"]     = True
    sim["ControlHDF5Content/WriteDiffusedPSF"]       = True
    sim["ControlHDF5Content/WriteHighResolutionPSF"] = True

    # Create star catalogue (path to starcat is automatically by function)
    xpos = np.array([int(xsub/2.) + xzero])
    ypos = np.array([int(ysub/2.) + yzero])
    mag  = np.array([10.0])
    ID   = np.array([1])
    starcat = f"{os.getcwd()}/starcat.txt"
    sim.createStarCatalogFileFromPixelCoordinates(xpos, ypos, mag, ID, starcat)

    # Run simulation
    sim.run(removeOutputFile=True)

In [ ]:
# Plot Zemax high res PSFs
fig, ax = plt.subplots(8, 8, sharex=True, sharey=True, figsize=(14,14))
fig.subplots_adjust(hspace=0.02, wspace=0)
fig.suptitle("Zemax High Resolution PSFs", y=0.9, fontsize=20)

# Create plots
for i, axes in zip(range(n), ax.flat):

    # Load high resolution and diffused PSFs
    alpha_i = alpha[i]
    inrot_i = inrot[i]
    string = "output_ZemaxPSF_num" + f"{i}".zfill(2) + "_alpha{:.1f}_inrot{:.1f}".format(alpha_i, inrot_i)
    f   = h5py.File(f'{os.getcwd()}/{string}.hdf5', 'r')
    psf = np.array(f["PSF/rotatedPSF"])[2*res_z:6*res_z, 2*res_z:6*res_z]  # We only want the inner 4 x 4 pixels
    
    # Plot PSF
    im = axes.imshow(psf, cmap=cm, interpolation='nearest', origin='lower')
    
    # Setting
    text = f"{alpha_i}" + r"$^{\circ}$, " + f"{inrot_i}" + r"$^{\circ}$"
    axes.text(0.5, 0.9, text, color='white', fontsize=13, 
              horizontalalignment='center', verticalalignment='center', transform=axes.transAxes)
    axes.set_xticks([])
    axes.set_yticks([])
    axes.set_xticklabels([])
    axes.set_yticklabels([])
    
    
# Save figure
fig.savefig('psfZemaxHigResFP.pdf', bbox_inches='tight', dpi=200)

In [ ]:
# Plot Zemax high res PSFs
fig, ax = plt.subplots(8, 8, sharex=True, sharey=True, figsize=(14,14))
fig.subplots_adjust(hspace=0.02, wspace=0)
fig.suptitle("Zemax High Resolution Diffused PSFs", y=0.9, fontsize=20)

# Create plots
for i, axes in zip(range(n), ax.flat):

    # Load high resolution and diffused PSFs
    alpha_i = alpha[i]
    inrot_i = inrot[i]
    string = "output_ZemaxPSF_num" + f"{i}".zfill(2) + "_alpha{:.1f}_inrot{:.1f}".format(alpha_i, inrot_i)
    f   = h5py.File(f'{os.getcwd()}/{string}.hdf5', 'r')
    psf = np.array(f["PSF/diffusedPSF"])[2*res_z:6*res_z, 2*res_z:6*res_z]  # We only want the inner 4 x 4 pixels
    
    # Plot PSF
    im = axes.imshow(psf, cmap=cm, interpolation='nearest', origin='lower')
    
    # Settings
    text = f"{alpha_i}" + r"$^{\circ}$, " + f"{inrot_i}" + r"$^{\circ}$"
    axes.text(0.5, 0.9, text, color='white', fontsize=13, 
              horizontalalignment='center', verticalalignment='center', transform=axes.transAxes)
    axes.set_xticks([])
    axes.set_yticks([])
    axes.set_xticklabels([])
    axes.set_yticklabels([])
    
# Save figure
fig.savefig('psfZemaxDiffusedFP.pdf', bbox_inches='tight', dpi=200)

In [ ]:
# Loop over each simulation
for i in tqdm(range(n), bar_format=ut.tqdm_bar_format()):

    # Select values
    alpha_i = alpha[i]
    inrot_i = inrot[i]
    
    # Initialise PlatoSim
    outputFileName = ("output_AnalyticPSF_num" + f"{i}".zfill(2) + 
                      "_alpha{:.1f}_inrot{:.1f}".format(alpha_i, inrot_i))
    sim = Simulation(outputFileName, outputDir=os.getcwd())

    # From alpha to focal plane coordinates
    focalLength = sim["Camera/FocalLength/ConstantValue"] * 1000
    xFP, yFP = rf.focalPlaneCoordinatesFromGnomonicRadialDistance(np.deg2rad(alpha_i), focalLength, 
                                                                  inPlaneRotation=np.deg2rad(inrot_i))

    # From focal plane coordinates to CCD coordinates
    pixelSize  = sim["CCD/PixelSize"]
    xCCD, yCCD = rf.focalPlaneToPixelCoordinates(xFP, yFP, pixelSize, ccdZeroPointX=0, ccdZeroPointY=0, CCDangle=0)

    # Observation
    sim["ObservingParameters/NumExposures"] = 1

    # Subfield (note y here is row and x is column)
    xzero = sim["SubField/ZeroPointRow"]    = int(yCCD)
    yzero = sim["SubField/ZeroPointColumn"] = int(xCCD)
    xsub  = sim["SubField/NumColumns"]      = 8
    ysub  = sim["SubField/NumRows"]         = 8

    # Camera
    sim["Camera/IncludePointLikeGhosts"] = False
    
    # PSF
    sim["PSF/Model"] = "AnalyticNonGaussian"
    sim["PSF/MappedFromFile/IncludeChargeDiffusion"] = True

    # Control HDF5
    sim["ControlHDF5Content/WriteStarPositions"]     = True
    sim["ControlHDF5Content/WriteDiffusedPSF"]       = True
    sim["ControlHDF5Content/WriteHighResolutionPSF"] = True

    # Create star catalogue (path to starcat is automatically by function)
    xpos = np.array([int(xsub/2.) + xzero])
    ypos = np.array([int(ysub/2.) + yzero])
    mag  = np.array([10.0])
    ID   = np.array([1])
    starcat = f"{os.getcwd()}/starcat_AnalyticPSF_matrix.txt"
    sim.createStarCatalogFileFromPixelCoordinates(xpos, ypos, mag, ID, starcat)

    # Run simulation
    sim.run(removeOutputFile=True)

In [ ]:
# Plot large view of PSFs
fig, ax = plt.subplots(8, 8, sharex=True, sharey=True, figsize=(14,14))
fig.subplots_adjust(hspace=0.02, wspace=0)
fig.suptitle("Analytic non-Gaussian High Resolution PSFs", y=0.9, fontsize=20)

# Create plots
for i, axes in zip(range(n), ax.flat):

    # Load high resolution and diffused PSFs
    alpha_i = alpha[i]
    inrot_i = inrot[i]
    string = "output_AnalyticPSF_num" + f"{i}".zfill(2) + "_alpha{:.1f}_inrot{:.1f}".format(alpha_i, inrot_i)
    f   = h5py.File(f'{os.getcwd()}/{string}.hdf5', 'r')
    psf = np.array(f["PSF/highResPSF"])[2*res_a:6*res_a, 2*res_a:6*res_a]  # We only want the inner 4 x 4 pixels
    
    # Plot PSF
    im = axes.imshow(psf, cmap=cm, interpolation='nearest', origin='lower')
    
    # Settings
    text = f"{alpha_i}" + r"$^{\circ}$, " + f"{inrot_i}" + r"$^{\circ}$"
    axes.text(0.5, 0.9, text, color='white', fontsize=13, 
              horizontalalignment='center', verticalalignment='center', transform=axes.transAxes)
    axes.set_xticks([])
    axes.set_yticks([])
    axes.set_xticklabels([])
    axes.set_yticklabels([])
    
# Save figure
fig.savefig('psfAnalyticHigResFP.pdf', bbox_inches='tight', dpi=200)

In [ ]:
# Plot large view of PSFs
fig, ax = plt.subplots(8, 8, sharex=True, sharey=True, figsize=(14,14))
fig.subplots_adjust(hspace=0.02, wspace=0)
fig.suptitle("Analytic non-Gaussian High Resolution Diffused PSFs", y=0.9, fontsize=20)

# Create plots
for i, axes in zip(range(n), ax.flat):

    # Load high resolution and diffused PSFs
    alpha_i = alpha[i]
    inrot_i = inrot[i]
    string = "output_AnalyticPSF_num" + f"{i}".zfill(2) + "_alpha{:.1f}_inrot{:.1f}".format(alpha_i, inrot_i)
    f   = h5py.File(f'{os.getcwd()}/{string}.hdf5', 'r')
    psf = np.array(f["PSF/diffusedPSF"])[2*res_a:6*res_a, 2*res_a:6*res_a]  # We only want the inner 4 x 4 pixels
    
    # Plot PSF
    im = axes.imshow(psf, cmap=cm, interpolation='nearest', origin='lower')
    
    # Settings
    text = f"{alpha_i}" + r"$^{\circ}$, " + f"{inrot_i}" + r"$^{\circ}$"
    axes.text(0.5, 0.9, text, color='white', fontsize=13, 
              horizontalalignment='center', verticalalignment='center', transform=axes.transAxes)
    axes.set_xticks([])
    axes.set_yticks([])
    axes.set_xticklabels([])
    axes.set_yticklabels([])
    
# Save figure
fig.savefig('psfAnalyticDiffusedFP.pdf', bbox_inches='tight', dpi=200)

### Residuals for diffused PSFs

In [ ]:
# Plot Zemax high res PSFs
fig, ax = plt.subplots(8, 8, sharex=True, sharey=True, figsize=(14,14))
fig.subplots_adjust(hspace=0.02, wspace=0)
fig.suptitle("Residuals between non-diffused PSFs: Analytic non-Gaussian - Zemax PSFs", y=0.9, fontsize=20)


# Create plots
for i, axes in zip(range(n), ax.flat):

    # Load high resolution and diffused PSFs
    alpha_i = alpha[i]
    inrot_i = inrot[i]
    
    # Zemax
    string_z = "output_ZemaxPSF_num" + f"{i}".zfill(2) + "_alpha{:.1f}_inrot{:.1f}".format(alpha_i, inrot_i)
    f_z      = h5py.File(f'{os.getcwd()}/{string_z}.hdf5', 'r')
    psf_z    = np.array(f_z["PSF/rotatedPSF"]) #[2*res_z:6*res_z, 2*res_z:6*res_z]  # We only want the inner 4 x 4 pixels
        
    # Analytic non-Gaussian
    string_a = "output_AnalyticPSF_num" + f"{i}".zfill(2) + "_alpha{:.1f}_inrot{:.1f}".format(alpha_i, inrot_i)
    f_a      = h5py.File(f'{os.getcwd()}/{string_a}.hdf5', 'r')
    psf_a    = np.array(f_a["PSF/highResPSF"]) #[2*res_a:6*res_a, 2*res_a:6*res_a]  # We only want the inner 4
    
    # Interpolate to same grid
    n_z = len(psf_z)
    n_a = len(psf_a)
    xx_a = yy_a = np.arange(n_a)
    xx_z = yy_z = xx_a[::2]
    interp = interp2d(xx_a, yy_a, psf_a, kind="cubic")
    psf_a = interp(xx_z, yy_z) * 4 # 4 is to preserve the sum to be 1

    # Residuals
    res = psf_a - psf_z
    res_zoom = res[2*res_z:6*res_z, 2*res_z:6*res_z]
    
    # Plot PSF
    im = axes.imshow(res_zoom, norm=colors.CenteredNorm(), cmap="PuOr", interpolation='nearest', origin='lower')
    
    # Settings
    text1 = f"{alpha_i}" + r"$^{\circ}$, " + f"{inrot_i}" + r"$^{\circ}$"
#     text2 = f"{res.min()*100:.2f}\%, {res.max()*100:.2f}\%"
    text2 = f"{res.min()*1e5:.1f}" + r"\textperthousand" + ", " + f"{res.max()*1e5:.1f}" + r"\textperthousand"

    axes.text(0.5, 0.9, text1, color='black', fontsize=13, 
              horizontalalignment='center', verticalalignment='center', transform=axes.transAxes)
    axes.text(0.5, 0.1, text2, color='black', fontsize=13, 
              horizontalalignment='center', verticalalignment='center', transform=axes.transAxes)
    
    axes.set_xticks([])
    axes.set_yticks([])
    axes.set_xticklabels([])
    axes.set_yticklabels([])
    
# Save figure
fig.savefig('psfResidualsHighResFP.pdf', bbox_inches='tight', dpi=200)

In [ ]:
# Plot Zemax high res PSFs
fig, ax = plt.subplots(8, 8, sharex=True, sharey=True, figsize=(14,14))
fig.subplots_adjust(hspace=0.02, wspace=0)
fig.suptitle("Residuals between Diffused PSFs: Analytic non-Gaussian - Zemax", y=0.9, fontsize=20)


# Create plots
for i, axes in zip(range(n), ax.flat):

    # Load high resolution and diffused PSFs
    alpha_i = alpha[i]
    inrot_i = inrot[i]
    
    # Zemax
    string_z = "output_ZemaxPSF_num" + f"{i}".zfill(2) + "_alpha{:.1f}_inrot{:.1f}".format(alpha_i, inrot_i)
    f_z      = h5py.File(f'{os.getcwd()}/{string_z}.hdf5', 'r')
    psf_z    = np.array(f_z["PSF/diffusedPSF"]) #[2*res_z:6*res_z, 2*res_z:6*res_z]  # We only want the inner 4 x 4 pixels
        
    # Analytic non-Gaussian
    string_a = "output_AnalyticPSF_num" + f"{i}".zfill(2) + "_alpha{:.1f}_inrot{:.1f}".format(alpha_i, inrot_i)
    f_a      = h5py.File(f'{os.getcwd()}/{string_a}.hdf5', 'r')
    psf_a    = np.array(f_a["PSF/diffusedPSF"]) #[2*res_a:6*res_a, 2*res_a:6*res_a]  # We only want the inner 4
    
    flux0 = psf_a.sum()
    
    # Interpolate to same grid
    n_z = len(psf_z)
    n_a = len(psf_a)
    xx_a = yy_a = np.arange(n_a)
    xx_z = yy_z = xx_a[::2]
    interp = interp2d(xx_a, yy_a, psf_a, kind="cubic")
    psf_a = interp(xx_z, yy_z) * 4 # 4 is to preserve the sum to be 1
    
    # Residuals
    res = psf_a - psf_z
    res_zoom = res[2*res_z:6*res_z, 2*res_z:6*res_z]
    
    # Plot PSF
    im = axes.imshow(res, norm=colors.CenteredNorm(), cmap="BrBG", interpolation='nearest', origin='lower')
    
    # Settings
    text1 = f"{alpha_i}" + r"$^{\circ}$, " + f"{inrot_i}" + r"$^{\circ}$"
#     text2 = f"{res.min()*1e5:.1f}"+ r"$\text{\textperthousand}$, " + f"{res.max()*1e5:.1f}\%"
    text2 = f"{res.min()*1e5:.1f}" + r"\textperthousand" + ", " + f"{res.max()*1e5:.1f}" + r"\textperthousand"

    axes.text(0.5, 0.9, text1, color='black', fontsize=13, 
              horizontalalignment='center', verticalalignment='center', transform=axes.transAxes)
    axes.text(0.5, 0.1, text2, color='black', fontsize=13, 
              horizontalalignment='center', verticalalignment='center', transform=axes.transAxes)
    
    axes.set_xticks([])
    axes.set_yticks([])
    axes.set_xticklabels([])
    axes.set_yticklabels([])
    
# Save figure
fig.savefig('psfResidualsDiffusedFP.pdf', bbox_inches='tight', dpi=200)